<h1>Overview</h1>
This notebook implements a CrewAI-based e-commerce customer-support triage pipeline. It uses four specialized agents arranged in sequence to analyze a customer complaint, apply policy rules, recommend a resolution, and decide whether human escalation is required.

<h1>Step 0 - Installation</h1>
Run the cell below once to install the required packages.

In [1]:
#pip install crewai==0.28.8 crewai_tools==0.1.6

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74/74 [crewai]ai] [crewai_tools]33m  WARNING: The script ec is installed in '/voc/work/.local/bin' which is not on PATH.  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.0mhon3.10/site-packages (from crewai==0.28.8) (1.4.4)Requirement already satisfied: click<9.0.0,>=8.1.7 in /usr/local/lib/python3.10/site-packages (from crewai==0.28.8) (8.1.7)Collecting embedchain<0.2.0,>=0.1.98 (from crewai==0.28.8)  Downloading embedchain-0.1.128-py3-none-any.whl.metadata (9.2 kB)Collecting instructor<0.6.0,>=0.5.2 (from crewai==0.28.8)  Downloading instructor-0.5.2-py3-none-any.whl.metadata (10 kB)Collecting langchain<0.2.0,>=0.1.10 (from crewai==0.28.8)  Downloading langchain-0.1.20-py3-none-any.whl.metadata (13 kB)Requirement already satisfied: openai<2.0.0,>=1.13.3 in /usr/local/lib/python3.10/site-packages (from crewai==0.28.8) (1.100.2)Requirement already satisfied: opentelemetry-api<2.0.0

<h1>Step 1 - Imports and LLM Initialisation</h1>
Here we:

1. Suppress noisy warnings for cleaner notebook output.
2. Load the OpenAI API key from a .env file (never hard-code secrets!).
3. Create a shared LLM object (gpt-4o-mini) that every agent will use.

temperature=0 makes responses deterministic - ideal for medical decision logic where you want consistent, repeatable outputs rather than creative variation.

In [13]:
# Warning control
import warnings
warnings.filterwarnings('ignore')
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
# Load environment variables
load_dotenv()

import warnings
warnings.filterwarnings('ignore')

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [14]:
POLICY_TEXT = '''
1. Returns are allowed within 30 days of delivery for returnable categories.
2. Damaged or wrong items reported within 7 days are eligible for refund or replacement.
3. Digital goods, perishables, and intimate items are non-returnable unless damaged or incorrect.
4. Delivered-but-not-received claims require escalation if delivery proof exists or order value exceeds 500.
5. Any low-confidence classification, missing order identifier, repeat refund pattern, or exception request must be escalated.
'''
print(POLICY_TEXT)


ESCALATION_RULES = [
    'Escalate if order value is above 500.',
    'Escalate if the order ID is missing or confidence is low.',
    'Escalate delivered-but-not-received claims with stronger risk signals.',
    'Escalate repeat refund patterns or suspicious timing.',
    'Escalate exceptions such as non-returnable but damaged items or urgent regulated/perishable cases.'
]

for rule in ESCALATION_RULES:
    print('-', rule)


1. Returns are allowed within 30 days of delivery for returnable categories.
2. Damaged or wrong items reported within 7 days are eligible for refund or replacement.
3. Digital goods, perishables, and intimate items are non-returnable unless damaged or incorrect.
4. Delivered-but-not-received claims require escalation if delivery proof exists or order value exceeds 500.
5. Any low-confidence classification, missing order identifier, repeat refund pattern, or exception request must be escalated.

- Escalate if order value is above 500.
- Escalate if the order ID is missing or confidence is low.
- Escalate delivered-but-not-received claims with stronger risk signals.
- Escalate repeat refund patterns or suspicious timing.
- Escalate exceptions such as non-returnable but damaged items or urgent regulated/perishable cases.


<h1> Step 2 - Import CrewAI Building Blocks </h1>

In [0]:
from crewai import Agent, Task, Crew

<h1>Step 3 - Define the Four Triage Agents</h1>

Each agent is built with four key parameters:

<table><b>role --></b>	   Job title - anchors the LLM to a professional identity</table>
<table><b>goal --></b>	    The agent's primary objective for this run</table>
<table><b>backstory --></b>	Extra context that sharpens the LLM's persona</table>
<table><b>llm --></b>	The     language model powering this agent</table>

In [15]:
agent_summary = {
    'Order Issue Identification Agent': 'Extracts facts, classifies issue type, estimates urgency and confidence.',
    'Policy Interpretation Agent': 'Applies static return, refund, and exception rules.',
    'Resolution Recommendation Agent': 'Recommends the best next action allowed by policy.',
    'Escalation Agent': 'Determines whether the case must be routed to a human.'
}

for name, purpose in agent_summary.items():
    print(f'- {name}: {purpose}')

# ----------------------
# Agents
# ----------------------

triage_agent = Agent(
    role='Order Issue Identification Agent',
    goal='Classify customer order problems and produce a structured issue case from customer text and mock order data.',
    backstory='Expert in e-commerce support triage. Extracts only the facts needed for policy and resolution decisions.',
    allow_delegation=False,
    verbose=True
)

policy_agent = Agent(
    role='Policy Interpretation Agent',
    goal='Apply static return, refund, and exception rules consistently to the structured case.',
    backstory='Expert in e-commerce operations policy and exception handling.',
    allow_delegation=False,
    verbose=True
)

resolution_agent = Agent(
    role='Resolution Recommendation Agent',
    goal='Recommend the best policy-compliant resolution with clear customer rationale and SLA.',
    backstory='Expert in balancing customer satisfaction with policy consistency and operational efficiency.',
    allow_delegation=False,
    verbose=True
)

escalation_agent = Agent(
    role='Escalation Agent',
    goal='Determine whether the case requires human review based on risk, urgency, fraud indicators, or uncertainty.',
    backstory='Senior support controller for high-risk and exception cases.',
    allow_delegation=False,
    verbose=True
)

print('All four agents are ready.')

- Order Issue Identification Agent: Extracts facts, classifies issue type, estimates urgency and confidence.
- Policy Interpretation Agent: Applies static return, refund, and exception rules.
- Resolution Recommendation Agent: Recommends the best next action allowed by policy.
- Escalation Agent: Determines whether the case must be routed to a human.
All four agents are ready.


<h1>Step 4 - Define Tasks </h1>
Each Task tells an agent <b>what to do</b> and <b>what output is expected.</b>

<b>Dynamic inputs with {}:</b> The {user_query} placeholder inside task descriptions is replaced at runtime when crew.kickoff(inputs={"user_query": "..."}) is called. This means you can run the exact same code for any user, just by changing the input string.

<b>Task chaining:</b> By default, CrewAI passes each task's output as context to the next task. So the Policy  agent automatically sees what the Triage agent found - no extra code needed.

In [47]:
triage_task = Task(
    description=f"""
    Customer query:
    {user_query}

    Identify the issue type (delay, refund, damage, return eligibility, wrong item, delivered-not-received, or other),
    infer urgency, extract facts, estimate confidence from 0 to 1, and flag any missing information or ambiguity.

    Return ONLY valid JSON matching IssueCase.
    """,
    expected_output="Structured issue case JSON.",
    agent=triage_agent
)

policy_task = Task(
        description=f'''
        Use the structured issue case and apply the following policy text:
        {POLICY_TEXT}

        Decide whether the request is eligible, what actions are allowed, and whether any exception flags apply.
        Return structured JSON matching PolicyDecision.
        ''',
        expected_output='Structured policy decision JSON.',
        agent=policy_agent,
        context=[triage_task]
    )

resolution_task = Task(
        description='''
        Using the issue case and policy decision, recommend one best next action.
        Candidate actions include: refund, replacement, reshipment, deny, ask for more info, partial refund, manual review.
        Include rationale and SLA. Return structured JSON matching ResolutionPlan.
        ''',
        expected_output='Structured resolution plan JSON.',
        agent=resolution_agent,
        context=[triage_task, policy_task]
    )

escalation_rules_text = "\n".join(f"- {rule}" for rule in ESCALATION_RULES)

escalation_task = Task(
    description=f"""
    You will receive the structured issue case, policy decision, and resolution plan.

    Apply these escalation rules:
    {escalation_rules_text}

    Escalate if:
    - order_value > 500
    - order_id is missing
    - confidence < 0.75
    - policy_exception is true
    - repeat_refund_pattern is true
    - delivered_not_received with risk signals

    Return ONLY valid JSON with:
    needs_escalation, reasons, priority, recommended_team, notes
    """,
    expected_output="Valid JSON matching EscalationDecision.",
    agent=escalation_agent,
    context=[triage_task, policy_task, resolution_task]
)

print('Tasks created')

Tasks created



<h1>Step 5 - Assemble the Crew (Pipeline) </h1>
Crew wires agents and tasks together into a sequential pipeline.

Tasks run in the order you list them.
Each agent automatically receives the previous agent's output as context.
<b>verbose=True</b> prints the full execution trace (great for learning and debugging).

The flow is:

triage_task -> policy_task -> resolution_task -> escalation_task

In [48]:
# ========================= CREW PIPELINE =========================

# Crew sequences the agents and tasks in the order provided.
# Each task's output is passed as context to the next task automatically.
triage_pipeline = Crew(
    agents=[triage_agent, policy_agent, resolution_agent, escalation_agent],
    tasks=[triage_task, policy_task, resolution_task, escalation_task],
    verbose=True    # Shows full agent reasoning; set to False for cleaner output
)

print("Crew pipeline assembled and ready to run.")

2026-05-01 10:32:17,828 - 139975280482112 - __init__.py-__init__:567 - WARNING: Overriding of current TracerProvider is not allowed


Crew pipeline assembled and ready to run.


<h1>Step 6 - Run the Triage Pipeline</h1>
Provide a plain-text patient description and call kickoff(). The pipeline injects user_query into all four tasks and runs them in sequence.

Change the text in user_query to any of the test cases at the bottom to see how the system responds to different severity levels.

In [21]:
# Sample user query - a critical, high-severity case with no manual intervention needed 
# This should trigger 'Escalation agent result
user_query = """
Customer issue: My headphones arrived cracked today and I want a refund or replacement.
Order ID: ORD-10452
Order value: 1299
Delivery status: Delivered today
Days since delivery: 0
Item category: Electronics
Previous refunds on account: 0
"""

# kickoff() starts the sequential pipeline and injects the input into all tasks
result = triage_pipeline.kickoff(inputs={"user_query": user_query})

print("\n" + "=" * 60)
print("FINAL TRIAGE OUTPUT")
print("=" * 60)
print(result)

 [DEBUG]: == Working Agent: Order Issue Identification Agent
 [INFO]: == Starting Task: 
    Customer query:
    
Customer issue: My headphones arrived cracked today and I want a refund or replacement.
Order ID: ORD-10452
Order value: 1299
Delivery status: Delivered today
Days since delivery: 0
Item category: Electronics
Previous refunds on account: 0


    Identify the issue type (delay, refund, damage, return eligibility, wrong item, delivered-not-received, or other),
    infer urgency, extract facts, estimate confidence from 0 to 1, and flag any missing information or ambiguity.

    Return ONLY valid JSON matching IssueCase.
    


> Entering new CrewAgentExecutor chain...
The customer's issue is related to a damaged product that they've received and they're asking for a refund or a replacement. They've provided all the necessary order details and there doesn't seem to be any ambiguity or missing information in their complaint. The problem seems urgent as the item was delivered tod

In [52]:
# Sample user query - potential high severity case
# This should trigger 'Escalation agent result
user_query = """
Issue: Delayed shipment
Message: My order is still in transit and already 5 days late. Can you help?
Order ID: ORD-77821
Order value: 89
Order date = 05 Apr 2026
Estimated delivery date = 10 Apr 2026
Delivery status: In transit
Days late: 5
Item category: Electronics
Previous refunds on account: 0
"""

# kickoff() starts the sequential pipeline and injects the input into all tasks
result = triage_pipeline.kickoff(inputs={"user_query": user_query})

print("\n" + "=" * 60)
print("FINAL TRIAGE OUTPUT")
print("=" * 60)
print(result)

 [DEBUG]: == Working Agent: Order Issue Identification Agent
 [INFO]: == Starting Task: 
    Customer query:
    
Return outside policy window.
I changed my mind and want to return the beauty product I received more than a month ago.
Order ID: ORD-88421
Order value: 349
Delivery status: Delivered 38 days ago
Days since delivery: 38
Item category: Beauty
Previous refunds on account: 0


    Identify the issue type (delay, refund, damage, return eligibility, wrong item, delivered-not-received, or other),
    infer urgency, extract facts, estimate confidence from 0 to 1, and flag any missing information or ambiguity.

    Return ONLY valid JSON matching IssueCase.
    


> Entering new CrewAgentExecutor chain...


BadRequestError: Error code: 400 - {'error': {'code': None, 'message': 'Insufficient budget available. Reason: Exceeded budget 1 > 1.0002', 'param': None, 'type': 'invalid_request_error'}}

In [49]:
# Sample user query - Outside return period item - request to be denied
# This should trigger 'Escalation agent result

user_query = """
Return outside policy window.
I changed my mind and want to return the beauty product I received more than a month ago.
Order ID: ORD-88421
Order value: 349
Delivery status: Delivered 38 days ago
Days since delivery: 38
Item category: Beauty
Previous refunds on account: 0
"""
# kickoff() starts the sequential pipeline and injects the input into all tasks
result = triage_pipeline.kickoff(inputs={"user_query": user_query})

print("\n" + "=" * 60)
print("FINAL TRIAGE OUTPUT")
print("=" * 60)
print(result)

 [DEBUG]: == Working Agent: Order Issue Identification Agent
 [INFO]: == Starting Task: 
    Customer query:
    
Return outside policy window.
I changed my mind and want to return the beauty product I received more than a month ago.
Order ID: ORD-88421
Order value: 349
Delivery status: Delivered 38 days ago
Days since delivery: 38
Item category: Beauty
Previous refunds on account: 0


    Identify the issue type (delay, refund, damage, return eligibility, wrong item, delivered-not-received, or other),
    infer urgency, extract facts, estimate confidence from 0 to 1, and flag any missing information or ambiguity.

    Return ONLY valid JSON matching IssueCase.
    


> Entering new CrewAgentExecutor chain...
The customer wants to return a beauty product which was delivered more than 30 days ago. The issue type is return eligibility as the customer wants to return the item outside the policy window. The urgency is low as the item has already been delivered and used. The facts are that 

# Test Cases - Try It Yourself
Replace the user_query string in Step 6 with any scenario below and re-run.


| # | Issue Description | Expected System Response |
| -------- | -------- | -------- |
| 1 | Simple return inquiry (within policy) | Eligibility confirmed, automated return label generated |
| 2 | Change of mind, outside 30-day window | Deny request, no escalation required |
| 3 | Order value > 500, missing delivery proof | High-risk flag + ESCALATE |
| 4 | Damaged item reported within 48 hours | Standard replacement process initiated |
| 5 | Potential refund abuse (multiple repeat claims) | Fraud alert + ESCALATE |